In [ ]:
%pip install webdriver-manager requests beautifulsoup4 lxml selenium pandas

In [ ]:
import os
import time
from dotenv import load_dotenv
import mysql.connector

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

# ---------------------------------------------------------
# DB 및 환경변수 설정
# ---------------------------------------------------------
load_dotenv(override=True)

DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'faq_data'), 
    'port': int(os.getenv('DB_PORT', 3306))
}

# DB 연결 및 커서 할당
conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()

insert_query = '''
INSERT INTO hyundai_faq (category_main, category_sub, question, answer) 
VALUES (%s, %s, %s, %s)
'''

# ---------------------------------------------------------
# 브라우저 초기화
# ---------------------------------------------------------
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized') 
options.add_argument('--window-size=1920,1080')

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 10)

url = 'https://www.hyundai.com/kr/ko/faq.html'
driver.get(url)
time.sleep(2)

try:
    # 드롭다운에서 대분류 리스트 추출
    main_select_elem = wait.until(EC.presence_of_element_located((By.ID, 'category_depth1')))
    main_select = Select(main_select_elem)
    main_options = [opt.text for opt in main_select.options if opt.get_attribute('value')]

    # ---------------------------------------------------------
    # 대분류별 크롤링 시작
    # ---------------------------------------------------------
    for main_text in main_options:
        try: 
            # 카테고리 선택 후 조회 버튼 클릭
            Select(driver.find_element(By.ID, 'category_depth1')).select_by_visible_text(main_text)
            time.sleep(1) 
            
            confirm_btn = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, '#contents > div.faq > div > div.section_white > div > form > fieldset > button'))
            )
            confirm_btn.click()
            
            page_num = 1
            while True:
                # 아코디언 리스트 렌더링 대기
                try:
                    WebDriverWait(driver, 5).until(
                        EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div.ui_accordion.acc_01 dl'))
                    )
                    time.sleep(1.5) # 컨텐츠 로딩 지연 대응
                except:
                    print(f"SKIP: [{main_text}] {page_num}페이지 로드 실패")
                    break
                
                soup = BeautifulSoup(driver.page_source, 'lxml')
                faq_list = soup.select('div.ui_accordion.acc_01 dl')
                
                page_data_batch = []

                for item in faq_list:
                    title_tag = item.select_one('.title')
                    content_tag = item.select_one('dd')
                    
                    if title_tag and content_tag:
                        q_raw = title_tag.text.strip()
                        
                        # 답변 내 이미지 태그를 텍스트(URL)로 치환 처리
                        images = content_tag.find_all('img')
                        for img in images:
                            img_src = img.get('src') or img.get('data-src')
                            if img_src:
                                # 상대 경로 대응 (도메인 결합)
                                if img_src.startswith('/'):
                                    img_src = 'https://www.hyundai.com' + img_src
                                elif not img_src.startswith('http'):
                                    img_src = 'https://www.hyundai.com/' + img_src
                                
                                cleaned_url = img_src.strip()
                                # 마크다운 파싱 에러 방지용 공백 추가
                                if cleaned_url.endswith('-') or cleaned_url.endswith('='):
                                    cleaned_url += ' '
                                
                                img.replace_with(f"\n\n[이미지: {cleaned_url}]\n\n")

                        # 구분자 포함 텍스트 추출 (줄바꿈 보존)
                        a_raw = content_tag.get_text(separator='\n', strip=True) 
                        
                        cat_main, cat_sub = main_text, "분류없음"
                        real_q = q_raw
                        
                        # [대분류 > 소분류] 질문 제목에서 카테고리 파싱
                        if '[' in q_raw and ']' in q_raw:
                            start_idx, end_idx = q_raw.find('['), q_raw.find(']')
                            category_part = q_raw[start_idx+1 : end_idx]
                            real_q = q_raw[end_idx+1 : ].strip()
                            
                            if '>' in category_part:
                                extracted_main, extracted_sub = category_part.split('>', 1)
                                cat_main, cat_sub = extracted_main.strip(), extracted_sub.strip()
                            else:
                                cat_main = category_part.strip()
                        
                        page_data_batch.append((cat_main, cat_sub, real_q, a_raw))
                
                # 페이지 단위 벌크 인서트
                if page_data_batch:
                    cursor.executemany(insert_query, page_data_batch)
                    conn.commit()

                print(f"DONE: [{main_text}] {page_num}p ({len(page_data_batch)}건)")

                # 다음 페이지 버튼 클릭 처리
                try:
                    next_btn = driver.find_element(By.CSS_SELECTOR, 'button.navi.next')
                    if next_btn.is_enabled() and next_btn.is_displayed():
                        driver.execute_script("arguments[0].click();", next_btn)
                        page_num += 1
                        time.sleep(1.5) 
                    else:
                        break 
                except:
                    break 

        except Exception as e:
            print(f"ERROR: [{main_text}] 루프 중 예외 발생 -> {e}")
            driver.get(url) # 에러 시 초기 페이지로 복귀
            time.sleep(2)
            continue

finally:
    # 자원 해제
    driver.quit()
    if cursor: cursor.close()
    if conn.is_connected(): conn.close()
    print("작업 종료: DB 커넥션 및 브라우저 닫기 완료")

In [ ]:
import os
import time
from dotenv import load_dotenv
import mysql.connector

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

# ---------------------------------------------------------
# DB 및 환경변수 로드
# ---------------------------------------------------------
load_dotenv(override=True)
DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'faq_data'), 
    'port': int(os.getenv('DB_PORT', 3306))
}

# DB 연결 및 커서 생성
conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()

insert_query = '''
INSERT INTO kia_faq (category, question, answer) 
VALUES (%s, %s, %s)
'''

# ---------------------------------------------------------
# 브라우저 초기 설정 (Selenium)
# ---------------------------------------------------------
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized') 
options.add_argument('--window-size=1920,1080')

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 10)

url = 'https://www.kia.com/kr/customer-service/center/faq'
driver.get(url)
time.sleep(3) 

# 수집 대상 카테고리 리스트
target_categories = ['차량 구매', '차량 정비', '기아멤버스', '홈페이지', 'PBV', '기타']

try:
    # ---------------------------------------------------------
    # 카테고리별 순회 루프
    # ---------------------------------------------------------
    for category in target_categories:
        try:
            print(f"\n>>> [{category}] 수집 시작")
            
            # 텍스트 매칭을 통한 카테고리 탭 클릭 (XPath 활용)
            tab_xpath = f"//*[contains(text(), '{category}') and (self::button or self::a or self::li or self::span)]"
            tab_elems = wait.until(EC.presence_of_all_elements_located((By.XPATH, tab_xpath)))
            
            clicked = False
            for elem in tab_elems:
                if elem.is_displayed(): 
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", elem)
                    time.sleep(0.5)
                    driver.execute_script("arguments[0].click();", elem)
                    clicked = True
                    break
            
            if not clicked:
                print(f"SKIP: [{category}] 탭 클릭 불가")
                continue
                
            time.sleep(2) 
            
            page_num = 1
            while True:
                # 아코디언 컴포넌트 렌더링 대기
                try:
                    WebDriverWait(driver, 5).until(
                        EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.cmp-accordion__title'))
                    )
                    time.sleep(1.5)
                except:
                    print(f"INFO: [{category}] 마지막 페이지 도달 ({page_num}p)")
                    break
                
                soup = BeautifulSoup(driver.page_source, 'lxml')
                titles = soup.select('.cmp-accordion__title') # 질문
                panels = soup.select('.cmp-accordion__panel') # 답변
                
                page_data_batch = []
                for t_tag, p_tag in zip(titles, panels):
                    q_text = t_tag.text.strip()
                    
                    # 답변 내 이미지 태그를 [이미지: URL] 형태의 텍스트로 치환
                    images = p_tag.find_all('img')
                    for img in images:
                        img_src = img.get('src')
                        if img_src:
                            # 절대 경로 보정
                            if img_src.startswith('/'):
                                img_src = 'https://www.kia.com' + img_src
                            elif not img_src.startswith('http'):
                                img_src = 'https://www.kia.com/' + img_src
                            
                            img.replace_with(f"\n\n[이미지: {img_src}]\n\n")

                    # 최종 텍스트 추출 (멀티라인 유지)
                    a_text = p_tag.get_text(separator='\n', strip=True)
                    page_data_batch.append((category, q_text, a_text))
                
                # 벌크 인서트 수행
                if page_data_batch:
                    cursor.executemany(insert_query, page_data_batch)
                    conn.commit()
                
                print(f"DONE: [{category}] {page_num}p 완료 ({len(page_data_batch)}건)")
                
                # ---------------------------------------------------------
                # 페이징 처리 (숫자 버튼 또는 '다음' 화살표)
                # ---------------------------------------------------------
                target_num = str(page_num + 1)
                num_xpath = f"//*[(self::a or self::button) and normalize-space()='{target_num}']"
                num_btns = driver.find_elements(By.XPATH, num_xpath)
                
                # 다음 숫자 버튼이 보이면 클릭
                if num_btns and num_btns[0].is_displayed():
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", num_btns[0])
                    time.sleep(0.5)
                    driver.execute_script("arguments[0].click();", num_btns[0])
                    page_num += 1
                    time.sleep(1.5)
                    continue
                else:
                    # 화살표 버튼(Next) 검색 및 클릭
                    next_arrow_xpath = "//*[(self::a or self::button) and (contains(@class, 'next') or contains(@aria-label, '다음'))]"
                    arrows = driver.find_elements(By.XPATH, next_arrow_xpath)
                    
                    clicked_arrow = False
                    for arrow in arrows:
                        if arrow.is_displayed() and arrow.is_enabled() and 'disabled' not in arrow.get_attribute('class'):
                            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", arrow)
                            time.sleep(0.5)
                            driver.execute_script("arguments[0].click();", arrow)
                            clicked_arrow = True
                            time.sleep(2)
                            break
                    
                    if clicked_arrow:
                        # 화살표 클릭 후 다시 숫자 버튼 체크
                        check_btns = driver.find_elements(By.XPATH, num_xpath)
                        if check_btns and check_btns[0].is_displayed():
                            driver.execute_script("arguments[0].click();", check_btns[0])
                            page_num += 1
                            time.sleep(1.5)
                            continue
                        else:
                            break 
                    else:
                        break 

        except Exception as e:
            print(f"ERROR: [{category}] 수집 중 중단 -> {e}")
            driver.get(url) # 섹션 에러 시 새로고침 후 다음 카테고리로 진행
            time.sleep(3)
            continue

finally:
    # 자원 정리
    driver.quit()
    if cursor: cursor.close()
    if conn.is_connected(): conn.close()
    print("\n[시스템] 크롤링 종료 및 DB 연결 해제 완료")

In [ ]:
import os
import time
from dotenv import load_dotenv
import mysql.connector

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

# ---------------------------------------------------------
# DB 및 환경 설정
# ---------------------------------------------------------
load_dotenv(override=True)
DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'faq_data'), 
    'port': int(os.getenv('DB_PORT', 3306))
}

# DB 연결 및 쿼리 준비
conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()

insert_query = '''
INSERT INTO hipass_faq (category_main, category_sub, question, answer) 
VALUES (%s, %s, %s, %s)
'''

# ---------------------------------------------------------
# 브라우저 초기화
# ---------------------------------------------------------
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized') 

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 10)

url = 'https://hipass.co.kr/board/selectFaqList.do'
driver.get(url)
time.sleep(3) 

try:
    print("\n>>> 하이패스 FAQ 전체 수집 시작")
    
    page_num = 1
    while True:
        # 리스트 테이블 렌더링 대기
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
        time.sleep(1) 
        
        rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
        row_count = len(rows)
        
        # 게시글이 없거나 검색 결과가 없는 경우 종료
        if row_count == 0 or (row_count == 1 and "없습니다" in rows[0].text):
            break 
            
        print(f"PROGRESS: {page_num}페이지 수집 중... ({row_count}개 항목)")

        # ---------------------------------------------------------
        # 상세 페이지 왕복 수집 루프
        # ---------------------------------------------------------
        for i in range(row_count):
            try:
                # 리스트 행 재참조 (뒤로가기 후 요소 소실 방지)
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
                current_rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
                row = current_rows[i]
                
                tds = row.find_elements(By.TAG_NAME, "td")
                if len(tds) < 4: continue
                
                main_cat = tds[1].text.strip() # 대분류
                sub_cat = tds[2].text.strip()  # 소분류
                
                subject_td = tds[3]
                question_text = subject_td.text.strip()
                
                # 상세 페이지 진입 (JS 클릭)
                subject_link = subject_td.find_element(By.TAG_NAME, "a")
                driver.execute_script("arguments[0].click();", subject_link)
                
                # 컨텐츠 영역 로딩 대기
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.cm_board_view_contents")))
                time.sleep(0.5)
                
                soup = BeautifulSoup(driver.page_source, 'lxml')
                answer_box = soup.select_one("div.cm_board_view_contents")
                
                if answer_box:
                    # 답변 내 이미지 태그를 [이미지: URL] 텍스트로 치환 (위치 보존)
                    images = answer_box.find_all('img')
                    for img in images:
                        img_src = img.get('src')
                        if img_src:
                            # 절대 경로 보정
                            if img_src.startswith('/'):
                                img_src = 'https://hipass.co.kr' + img_src
                            elif not img_src.startswith('http'):
                                img_src = 'https://hipass.co.kr/' + img_src
                            
                            img.replace_with(f"\n\n[이미지: {img_src}]\n\n")
                    
                    # 최종 텍스트 추출 (멀티라인 유지)
                    answer_text = answer_box.get_text(separator='\n', strip=True)
                else:
                    answer_text = "내용 없음"
                
                # DB 데이터 적재
                cursor.execute(insert_query, (main_cat, sub_cat, question_text, answer_text))
                conn.commit()
                
                # 목록으로 돌아가기
                driver.back()
                time.sleep(1) 
                
                # 뒤로가기 시 페이지 번호가 틀어지는 경우를 대비한 보정 로직
                active_page_elem = driver.find_elements(By.CSS_SELECTOR, "a.page_link.active")
                if active_page_elem and active_page_elem[0].text.strip() != str(page_num):
                    driver.execute_script(f"movePage('{page_num}', '/board/selectFaqList.do')")
                    time.sleep(1.5)

            except Exception as e:
                print(f"WARN: {i+1}번째 글 처리 중 오류 발생 (Skip) -> {e}")
                driver.get(url)
                driver.execute_script(f"movePage('{page_num}', '/board/selectFaqList.do')")
                time.sleep(2)
                continue

        print(f"DONE: {page_num}페이지 적재 완료")

        # ---------------------------------------------------------
        # 다음 페이지로 이동 처리
        # ---------------------------------------------------------
        target_num_str = str(page_num + 1)
        # 현재 블록 내 다음 숫자 버튼 검색
        page_links = driver.find_elements(By.XPATH, f"//a[contains(@class, 'page_link') and text()='{target_num_str}']")

        if page_links:
            driver.execute_script("arguments[0].click();", page_links[0])
            page_num += 1
            time.sleep(1.5)
        else:
            # 다음 숫자 버튼이 없으면 '다음' 화살표 버튼 확인
            next_arrow = driver.find_elements(By.CSS_SELECTOR, "a.page_link.page_control.next")
            if next_arrow and next_arrow[0].is_displayed():
                driver.execute_script("arguments[0].click();", next_arrow[0])
                page_num += 1
                time.sleep(1.5)
            else:
                break # 다음 페이지 없음

except Exception as e:
    print(f"FATAL ERROR: 크롤링 중단 -> {e}")

finally:
    # 자원 반납
    driver.quit()
    if cursor: cursor.close()
    if conn.is_connected(): conn.close()
    print("\n[시스템] 하이패스 크롤러 종료 및 세션 정리 완료")